In [55]:

import re
import os
import json
import uuid
import json
import time
import spacy
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from typing import Dict, Any, Optional


os.getcwd()

'c:\\Users\\maina\\Documents\\kdd-project'

# Loading the job dataset and preprocessing

In [14]:
# Columns we want to keep are: job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips
# We will have zip_code and fips as strings instead of integers because they can have leading zeros, job_id and company_id as strings
# and we dont want to lose that information. Also, we have everything to lowercase to avoid any issues with case sensitivity when we do our analysis later on.

columns_to_keep = ['job_id', 'company_name', 'title', 'description', 'location', 'company_id', 'views', 'skills_desc', 'work_type', 'zip_code', 'fips']

jobs = pd.read_csv('data/postings.csv',
                   usecols=columns_to_keep,
                   dtype={'job_id': str, 'company_id': str, 'zip_code': str, 'fips': str})


text_cols = ['company_name', 'title', 'description', 'location', 'skills_desc', 'work_type']

for col in text_cols:
    jobs[col] = jobs[col].str.lower().str.strip()

In [15]:
jobs.head()

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips
0,921716,corcoran sawyer smith,marketing coordinator,job descriptiona leading real estate firm in n...,"princeton, nj",2774458.0,20.0,requirements: \n\nwe are seeking a college or ...,full_time,08540,34021
1,1829192,NaN,mental health therapist/counselor,"at aspen therapy and wellness , we are committ...","fort collins, co",NaN,1.0,NaN,full_time,80521,08069
2,10998357,the national exemplar,assitant restaurant manager,the national exemplar is accepting application...,"cincinnati, oh",64896719.0,8.0,we are currently accepting resumes for foh - a...,full_time,45202,39061
3,23221523,"abrams fensterman, llp",senior elder law / trusts and estates associat...,senior associate attorney - elder law / trusts...,"new hyde park, ny",766262.0,16.0,this position requires a baseline understandin...,full_time,11040,36059
4,35982263,NaN,service technician,looking for hvac service tech with experience ...,"burlington, ia",NaN,3.0,NaN,full_time,52601,19057


In [16]:
jobs.tail()

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips
123844,3906267117,lozano smith,title ix/investigations attorney,our walnut creek office is currently seeking a...,"walnut creek, ca",56120.0,1.0,NaN,full_time,94595,06013
123845,3906267126,pinterest,"staff software engineer, ml serving platform",about pinterest:\n\nmillions of people across ...,united states,1124131.0,3.0,NaN,full_time,NaN,NaN
123846,3906267131,eps learning,"account executive, oregon/washington",company overview\n\neps learning is a leading ...,"spokane, wa",90552133.0,3.0,NaN,full_time,99201,53063
123847,3906267195,trelleborg applied technologies,business development manager,the business development manager is a 'hunter'...,"texas, united states",2793699.0,4.0,NaN,full_time,NaN,NaN
123848,3906267224,solugenix,marketing social media specialist,marketing social media specialist - $70k – $75...,"san juan capistrano, ca",43325.0,2.0,NaN,full_time,92675,06059


In [17]:
jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   job_id        123849 non-null  object 
 1   company_name  122130 non-null  object 
 2   title         123849 non-null  object 
 3   description   123842 non-null  object 
 4   location      123849 non-null  object 
 5   company_id    122132 non-null  object 
 6   views         122160 non-null  float64
 7   skills_desc   2439 non-null    object 
 8   work_type     123849 non-null  object 
 9   zip_code      102977 non-null  object 
 10  fips          96434 non-null   object 
dtypes: float64(1), object(10)
memory usage: 10.4+ MB


In [18]:
# We notice that some of the rows have country as united states , we dont want this. We have the zip_code and fips code which can help us identify the location of the job,
# We will use the zip_code and fips to get a city and state for each job, for rows where location is united states and no zip_code/fips we will drop those rows. 
# Normalize zip/fips as 5-digit strings (keep leading zeros)


# normalize location safely
jobs['location'] = (
    jobs['location']
    .astype(str)
    .str.strip()
    .str.lower()
)

# define "missing zip" robustly
missing_zip = (
    jobs['zip_code'].isna() |
    (jobs['zip_code'].astype(str).str.strip() == '') |
    (jobs['zip_code'].astype(str).str.lower() == 'nan')
)

mask_drop = (
    jobs['location'].eq('united states') &
    missing_zip &
    jobs['fips'].isna()
)

before = len(jobs)
jobs = jobs[~mask_drop]
after = len(jobs)

print(f"Dropped {before - after} rows")



Dropped 8125 rows


In [19]:
jobs.head(10)

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips
0,921716,corcoran sawyer smith,marketing coordinator,job descriptiona leading real estate firm in n...,"princeton, nj",2774458.0,20.0,requirements: \n\nwe are seeking a college or ...,full_time,08540,34021
1,1829192,NaN,mental health therapist/counselor,"at aspen therapy and wellness , we are committ...","fort collins, co",NaN,1.0,NaN,full_time,80521,08069
2,10998357,the national exemplar,assitant restaurant manager,the national exemplar is accepting application...,"cincinnati, oh",64896719.0,8.0,we are currently accepting resumes for foh - a...,full_time,45202,39061
3,23221523,"abrams fensterman, llp",senior elder law / trusts and estates associat...,senior associate attorney - elder law / trusts...,"new hyde park, ny",766262.0,16.0,this position requires a baseline understandin...,full_time,11040,36059
4,35982263,NaN,service technician,looking for hvac service tech with experience ...,"burlington, ia",NaN,3.0,NaN,full_time,52601,19057
5,91700727,downtown raleigh alliance,economic development and planning intern,job summary:the economic development & plannin...,"raleigh, nc",1481176.0,9.0,NaN,internship,27601,37183
7,112576855,NaN,building engineer,summary: due to the pending retirement of our ...,"san francisco, ca",NaN,2.0,NaN,full_time,94101,06075
8,1218575,children's nebraska,respiratory therapist,"at children’s, the region’s only full-service ...","omaha, ne",721189.0,3.0,• requires the ability to communicate effectiv...,full_time,68102,31055
9,2264355,bay west church,worship leader,it is an exciting time to be a part of our chu...,"palm bay, fl",28631247.0,5.0,"knowledge, skills and abilities: 1. proficient...",part_time,32905,12009
10,9615617,"glastender, inc.",inside customer service associate,glastender inc. is a family-owned manufacturer...,"saginaw, mi",1194336.0,4.0,the production supervisor must possess strong ...,full_time,48601,26145


In [20]:
jobs[jobs['location'] == 'united states'][['location', 'zip_code', 'fips']].head()

,location,zip_code,fips


In [21]:
zips_raw = pd.read_csv('data/uszips.csv')
zips_raw.columns = zips_raw.columns.str.strip().str.lower()

state_col = next(
    (c for c in ['state', 'state_id', 'state_name'] if c in zips_raw.columns),
    None
)

missing_required = [c for c in ['zip', 'city'] if c not in zips_raw.columns]
if missing_required:
    raise ValueError(f"Missing required columns in uszips.csv: {missing_required}")

if state_col is None:
    zips = zips_raw[['zip', 'city']].copy()
    zips['state'] = pd.NA
else:
    zips = zips_raw[['zip', 'city', state_col]].copy().rename(columns={state_col: 'state'})

zips['zip'] = zips['zip'].astype('string').str.strip().str.zfill(5)
jobs['zip_code'] = jobs['zip_code'].astype('string').str.strip().str.zfill(5)

# make reruns safe in notebook
jobs = jobs.drop(columns=['zip', 'city', 'state'], errors='ignore')

jobs = jobs.merge(
    zips,
    left_on='zip_code',
    right_on='zip',
    how='left'
)

In [22]:
jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115724 entries, 0 to 115723
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   job_id        115724 non-null  object 
 1   company_name  114283 non-null  object 
 2   title         115724 non-null  object 
 3   description   115720 non-null  object 
 4   location      115724 non-null  object 
 5   company_id    114285 non-null  object 
 6   views         114161 non-null  float64
 7   skills_desc   2409 non-null    object 
 8   work_type     115724 non-null  object 
 9   zip_code      102977 non-null  string 
 10  fips          96434 non-null   object 
 11  zip           99949 non-null   string 
 12  city          99949 non-null   object 
 13  state         99949 non-null   object 
dtypes: float64(1), object(11), string(2)
memory usage: 12.4+ MB


In [23]:
missing_city_state = jobs['city'].isna() & jobs['state'].isna()

In [24]:
STATE_MAP = {
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR',
    'california': 'CA', 'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE',
    'florida': 'FL', 'georgia': 'GA', 'hawaii': 'HI', 'idaho': 'ID',
    'illinois': 'IL', 'indiana': 'IN', 'iowa': 'IA', 'kansas': 'KS',
    'kentucky': 'KY', 'louisiana': 'LA', 'maine': 'ME', 'maryland': 'MD',
    'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN',
    'mississippi': 'MS', 'missouri': 'MO', 'montana': 'MT',
    'nebraska': 'NE', 'nevada': 'NV', 'new hampshire': 'NH',
    'new jersey': 'NJ', 'new mexico': 'NM', 'new york': 'NY',
    'north carolina': 'NC', 'north dakota': 'ND', 'ohio': 'OH',
    'oklahoma': 'OK', 'oregon': 'OR', 'pennsylvania': 'PA',
    'rhode island': 'RI', 'south carolina': 'SC', 'south dakota': 'SD',
    'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT', 'vermont': 'VT',
    'virginia': 'VA', 'washington': 'WA', 'west virginia': 'WV',
    'wisconsin': 'WI', 'wyoming': 'WY', 'district of columbia': 'DC'
}

In [25]:
def extract_state_from_location(loc):
    if not isinstance(loc, str):
        return None

    loc_lower = loc.lower()

    for name, abbr in STATE_MAP.items():
        abbr_lower = abbr.lower()

        # check full state name
        if name in loc_lower:
            return abbr

        # check abbreviation as a word boundary (tx, TX, Tx, etc.)
        pattern = rf'\b{abbr_lower}\b'
        if re.search(pattern, loc_lower):
            return abbr

    return None



jobs.loc[missing_city_state, 'state'] = (
    jobs.loc[missing_city_state, 'location']
    .apply(extract_state_from_location)
)

jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115724 entries, 0 to 115723
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   job_id        115724 non-null  object 
 1   company_name  114283 non-null  object 
 2   title         115724 non-null  object 
 3   description   115720 non-null  object 
 4   location      115724 non-null  object 
 5   company_id    114285 non-null  object 
 6   views         114161 non-null  float64
 7   skills_desc   2409 non-null    object 
 8   work_type     115724 non-null  object 
 9   zip_code      102977 non-null  string 
 10  fips          96434 non-null   object 
 11  zip           99949 non-null   string 
 12  city          99949 non-null   object 
 13  state         111748 non-null  object 
dtypes: float64(1), object(11), string(2)
memory usage: 12.4+ MB


In [26]:
jobs = jobs.drop(columns=['zip'], errors='ignore')

In [27]:
jobs.head()

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips,city,state
0,921716,corcoran sawyer smith,marketing coordinator,job descriptiona leading real estate firm in n...,"princeton, nj",2774458.0,20.0,requirements: \n\nwe are seeking a college or ...,full_time,08540,34021,Princeton,NJ
1,1829192,NaN,mental health therapist/counselor,"at aspen therapy and wellness , we are committ...","fort collins, co",NaN,1.0,NaN,full_time,80521,08069,Fort Collins,CO
2,10998357,the national exemplar,assitant restaurant manager,the national exemplar is accepting application...,"cincinnati, oh",64896719.0,8.0,we are currently accepting resumes for foh - a...,full_time,45202,39061,Cincinnati,OH
3,23221523,"abrams fensterman, llp",senior elder law / trusts and estates associat...,senior associate attorney - elder law / trusts...,"new hyde park, ny",766262.0,16.0,this position requires a baseline understandin...,full_time,11040,36059,New Hyde Park,NY
4,35982263,NaN,service technician,looking for hvac service tech with experience ...,"burlington, ia",NaN,3.0,NaN,full_time,52601,19057,Burlington,IA


In [28]:
# We drop the rows where state is NaN even after our best efforts to extract it from the location string, we will not be able to use those rows in our analysis and 
# they are a small percentage of the total data so we can afford to drop them.

jobs = jobs.dropna(subset=['state'])
jobs.info()


<class 'pandas.core.frame.DataFrame'>
Index: 111748 entries, 0 to 115723
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   job_id        111748 non-null  object 
 1   company_name  110422 non-null  object 
 2   title         111748 non-null  object 
 3   description   111745 non-null  object 
 4   location      111748 non-null  object 
 5   company_id    110424 non-null  object 
 6   views         110298 non-null  float64
 7   skills_desc   2310 non-null    object 
 8   work_type     111748 non-null  object 
 9   zip_code      102977 non-null  string 
 10  fips          96434 non-null   object 
 11  city          99949 non-null   object 
 12  state         111748 non-null  object 
dtypes: float64(1), object(11), string(1)
memory usage: 11.9+ MB


In [29]:
#  Use only 30% of cleaned real job dataset

jobs_sample = jobs.sample(n=1000, random_state=42).reset_index(drop=True)

# Lets also save the job_sample to a csv file. we will use it for synthetic data generation and testing our model later on.
jobs_sample.to_csv('data/jobs_sample.csv', index=False)

print("Original cleaned jobs shape:", jobs.shape)
print("sampled jobs shape:", jobs_sample.shape)


Original cleaned jobs shape: (111748, 13)
sampled jobs shape: (1000, 13)


In [30]:
jobs_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   job_id        1000 non-null   object 
 1   company_name  990 non-null    object 
 2   title         1000 non-null   object 
 3   description   1000 non-null   object 
 4   location      1000 non-null   object 
 5   company_id    990 non-null    object 
 6   views         985 non-null    float64
 7   skills_desc   19 non-null     object 
 8   work_type     1000 non-null   object 
 9   zip_code      933 non-null    string 
 10  fips          863 non-null    object 
 11  city          900 non-null    object 
 12  state         1000 non-null   object 
dtypes: float64(1), object(11), string(1)
memory usage: 101.7+ KB


In [31]:
jobs_sample.head()

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips,city,state
0,3903466305,cabela's,firearms outfitter - part time,position summary:\n\nthe sales outfitter - fir...,"lubbock, tx",12890.0,4.0,NaN,part_time,79401,48303,Lubbock,TX
1,3887485157,mission staffing,administrative assistant,"our client, a global law firm in washington, d...",washington dc-baltimore area,132319.0,32.0,NaN,full_time,<NA>,NaN,NaN,WA
2,3895598418,harbor freight tools,retail stocking supervisor,a supervisor (full-time) is a valued member of...,"chino hills, ca",28923.0,5.0,NaN,full_time,91709,06071,Chino Hills,CA
3,3906097453,bmo u.s.,"director, corporate development","reporting to vp/head of corporate development,...","chicago, il",164126.0,4.0,NaN,full_time,60601,17031,Chicago,IL
4,3903473957,university of south florida,general maintenance technician iii,department number/name: 0-0255-001 / maintenan...,"tampa, fl",166671.0,4.0,NaN,full_time,33602,12057,Tampa,FL


# Loading the student dataset and cleaning it

In [32]:
students_data = pd.read_csv("data\student_data.csv")
# lets convert all the columns to lowercase to avoid any issues with case sensitivity when we do our analysis later on.
str_cols = students_data.select_dtypes(include=["object", "string"]).columns
students_data[str_cols] = students_data[str_cols].apply(lambda col: col.str.lower())
students_data["Skill"] = (
    students_data["Skill"]
    .str.strip("[]")
    .str.replace('"', '')
)
students_data.head()

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\maina\AppData\Local\Temp\ipykernel_26768\4265789378.py:1: SyntaxWarning: invalid escape sequence '\s'
  students_data = pd.read_csv("data\student_data.csv")


,Student ID,Academic Major,GPA,Skill,Location Interest,University,School Year,Area of Experience,Number of Experience (yrs),Degree
0,stu000001,marketing,3.53,"client communication, market research, social ...",remote / flexible,boston university,sophomore,customer success,1.2,bachelor's
1,stu000002,finance,3.83,"collaboration, presentation skills, compliance...","seattle, wa",university of michigan,junior,risk management,1.6,bachelor's
2,stu000003,electrical engineering,3.70,"documentation, communication, verilog, problem...","washington, dc",university of pennsylvania,sophomore,electrical systems,0.9,bachelor's
3,stu000004,civil engineering,2.89,"project planning, civil 3d, attention to detai...","washington, dc",cornell university,senior,environmental engineering,3.2,bachelor's
4,stu000005,english,3.74,"editing, storytelling, content creation, prese...","san francisco bay area, ca",stanford university,freshman,communications,0.4,bachelor's


In [33]:
students_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student ID                  25000 non-null  object 
 1   Academic Major              25000 non-null  object 
 2   GPA                         25000 non-null  float64
 3   Skill                       25000 non-null  object 
 4   Location Interest           25000 non-null  object 
 5   University                  25000 non-null  object 
 6   School Year                 25000 non-null  object 
 7   Area of Experience          25000 non-null  object 
 8   Number of Experience (yrs)  25000 non-null  float64
 9   Degree                      25000 non-null  object 
dtypes: float64(2), object(8)
memory usage: 1.9+ MB


In [34]:
#  Keep only 50 student rows
students_sample = students_data.sample(n=50, random_state=42).reset_index(drop=True)

print("Original students_data shape:", students_data.shape)
print("Student sample shape:", students_sample.shape)


Original students_data shape: (25000, 10)
Student sample shape: (50, 10)


In [35]:
students_sample.head()

,Student ID,Academic Major,GPA,Skill,Location Interest,University,School Year,Area of Experience,Number of Experience (yrs),Degree
0,stu006869,computer science,3.20,"git, java, organization","seattle, wa",university of washington,sophomore,machine learning,0.8,bachelor's
1,stu024017,biology,3.57,"pcr, collaboration, presentation skills, spss,...","san diego, ca","university of california, san diego",2nd year master's,laboratory science,3.4,master's
2,stu009669,data science,3.57,"leadership, scikit-learn, problem solving, ini...","new york, ny",rutgers university,junior,machine learning,1.2,bachelor's
3,stu013641,economics,3.11,"policy analysis, adaptability, stata, r, prese...",remote / flexible,duke university,junior,finance,2.0,bachelor's
4,stu014019,business administration,3.23,"project management, sql, excel, presentation s...",remote / flexible,university of chicago,senior,finance,3.4,bachelor's


# Text Vectorization

In [36]:
def build_job_text(jobs_sample):
    jobs_sample = jobs_sample.copy()
    jobs_sample["description"] = jobs_sample["description"].fillna("")
    jobs_sample["job_text"] = (jobs_sample["title"] + " " + jobs_sample["description"]).str.strip()
    return jobs_sample

In [37]:
jobs_text = build_job_text(jobs_sample)
jobs_text.head()

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips,city,state,job_text
0,3903466305,cabela's,firearms outfitter - part time,position summary:\n\nthe sales outfitter - fir...,"lubbock, tx",12890.0,4.0,NaN,part_time,79401,48303,Lubbock,TX,firearms outfitter - part time position summar...
1,3887485157,mission staffing,administrative assistant,"our client, a global law firm in washington, d...",washington dc-baltimore area,132319.0,32.0,NaN,full_time,<NA>,NaN,NaN,WA,"administrative assistant our client, a global ..."
2,3895598418,harbor freight tools,retail stocking supervisor,a supervisor (full-time) is a valued member of...,"chino hills, ca",28923.0,5.0,NaN,full_time,91709,06071,Chino Hills,CA,retail stocking supervisor a supervisor (full-...
3,3906097453,bmo u.s.,"director, corporate development","reporting to vp/head of corporate development,...","chicago, il",164126.0,4.0,NaN,full_time,60601,17031,Chicago,IL,"director, corporate development reporting to v..."
4,3903473957,university of south florida,general maintenance technician iii,department number/name: 0-0255-001 / maintenan...,"tampa, fl",166671.0,4.0,NaN,full_time,33602,12057,Tampa,FL,general maintenance technician iii department ...


In [38]:
nlp = spacy.load("en_core_web_sm")

def normalize_title_spacy(title):
    doc = nlp(title.lower())
    
    # Keep nouns and proper nouns (core job meaning)
    tokens = [token.text for token in doc if token.pos_ in ["NOUN", "PROPN"]]
    
    return " ".join(tokens)

In [39]:
jobs_text["normalized_title"] = jobs_text["title"].apply(normalize_title_spacy)
jobs_text[["title", "normalized_title"]].head()

,title,normalized_title
0,firearms outfitter - part time,firearms outfitter part time
1,administrative assistant,assistant
2,retail stocking supervisor,stocking supervisor
3,"director, corporate development",director development
4,general maintenance technician iii,maintenance technician iii


# Bert Model Training on real job data

In [40]:
from sentence_transformers import SentenceTransformer

def vectorize_jobs_bert(jobs_text):
    model = SentenceTransformer('all-MiniLM-L6-v2')

    job_vectors = model.encode(
        jobs_text["job_text"].tolist(),
        show_progress_bar=True
    )

    return model, job_vectors

c:\Users\maina\Documents\kdd-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [41]:
model, job_vectors = vectorize_jobs_bert(jobs_text)

print("Job text shape:", jobs_text.shape)
print("Job vectors shape:", job_vectors.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2824.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 32/32 [00:39<00:00,  1.23s/it]

Job text shape: (1000, 15)
Job vectors shape: (1000, 384)


In [42]:
def build_student_text(students):
    students = students.copy()
    students["student_text"] = (
        students["Academic Major"].fillna("") + " " +
        students["Skill"].fillna("") + " " +
        students["Area of Experience"].fillna("") + " " +
        students["School Year"].fillna("") + " "+
        students["Degree"].fillna("")
    ).str.strip()
    return students

In [43]:
student_text = build_student_text(students_sample)
student_text.head()

,Student ID,Academic Major,GPA,Skill,Location Interest,University,School Year,Area of Experience,Number of Experience (yrs),Degree,student_text
0,stu006869,computer science,3.20,"git, java, organization","seattle, wa",university of washington,sophomore,machine learning,0.8,bachelor's,"computer science git, java, organization machi..."
1,stu024017,biology,3.57,"pcr, collaboration, presentation skills, spss,...","san diego, ca","university of california, san diego",2nd year master's,laboratory science,3.4,master's,"biology pcr, collaboration, presentation skill..."
2,stu009669,data science,3.57,"leadership, scikit-learn, problem solving, ini...","new york, ny",rutgers university,junior,machine learning,1.2,bachelor's,"data science leadership, scikit-learn, problem..."
3,stu013641,economics,3.11,"policy analysis, adaptability, stata, r, prese...",remote / flexible,duke university,junior,finance,2.0,bachelor's,"economics policy analysis, adaptability, stata..."
4,stu014019,business administration,3.23,"project management, sql, excel, presentation s...",remote / flexible,university of chicago,senior,finance,3.4,bachelor's,"business administration project management, sq..."


In [44]:
student_vectors = model.encode(
    student_text["student_text"].tolist(),
    show_progress_bar=True
)

print("Student text shape:", student_text.shape)
print("Student vectors shape:", student_vectors.shape)

Batches: 100%|██████████| 2/2 [00:00<00:00,  6.67it/s]

Student text shape: (50, 11)
Student vectors shape: (50, 384)


In [45]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(student_vectors, job_vectors)

print("Similarity matrix shape:", similarity_matrix.shape)

top_k = 5
top_matches = np.argsort(-similarity_matrix, axis=1)[:, :top_k]

results = []

for student_idx, job_idxs in enumerate(top_matches):
    for rank, job_idx in enumerate(job_idxs, start=1):
        results.append({
            "student_index": student_idx,
            #"student_text": student_text.iloc[student_idx]["student_text"],
            "job_index": job_idx,
            "job_id": jobs_text.iloc[job_idx]["job_id"],
            #"job_title": jobs_text.iloc[job_idx]["title"],
            "job_title_normalized": jobs_text.iloc[job_idx]["normalized_title"],
            #"job_text": jobs_text.iloc[job_idx]["job_text"],
            "similarity_score": similarity_matrix[student_idx, job_idx],
            "rank": rank
        })

matches_df = pd.DataFrame(results)
matches_df.head(10)

Similarity matrix shape: (50, 1000)


,student_index,job_index,job_id,job_title_normalized,similarity_score,rank
0,0,141,3901388774,java software engineer,0.552618,1
1,0,22,3884447074,developer,0.498012,2
2,0,582,3898175587,tibco bw ce lead,0.496360,3
3,0,182,3886830826,data scientist,0.496009,4
4,0,515,3898176128,associate stack engineer,0.469248,5
5,1,689,3906239463,technician,0.509629,1
6,1,912,3901958153,scientist ii flow cytometry,0.498239,2
7,1,588,3895538867,research associate ii,0.489958,3
8,1,690,3904964506,research engineer laboratory automation,0.428371,4
9,1,192,3902318887,sr sales development representative cryo life ...,0.402186,5


In [46]:
student_id_to_check = 10

top_jobs = matches_df[matches_df["student_index"] == student_id_to_check].copy()
top_jobs

,student_index,job_index,job_id,job_title_normalized,similarity_score,rank
50,10,547,3903461961,marketing manager,0.514762,1
51,10,252,3904920396,seo associate | bankrate,0.458467,2
52,10,258,3886826444,media marketing consultant work,0.454276,3
53,10,631,3891008383,director media,0.406619,4
54,10,717,3906099419,education specialist,0.404351,5


In [47]:
title_scores = (
    top_jobs.groupby("job_title_normalized", as_index=False)
    .agg(
        avg_similarity=("similarity_score", "mean"),
        retrieved_posting_count=("job_id", "count")
    )
    .rename(columns={"job_title_normalized": "title"})
)

title_scores["combined_score"] = (
    title_scores["avg_similarity"] *
    np.log1p(title_scores["retrieved_posting_count"])
)

title_scores = title_scores.sort_values("combined_score", ascending=False)

print("\nMatched titles:")
print(title_scores.head(10))


Matched titles:
                             title  avg_similarity  retrieved_posting_count  \
2                marketing manager        0.514762                        1   
4         seo associate | bankrate        0.458467                        1   
3  media marketing consultant work        0.454276                        1   
0                   director media        0.406619                        1   
1             education specialist        0.404351                        1   

   combined_score  
2        0.356806  
4        0.317785  
3        0.314880  
0        0.281847  
1        0.280275  


In [48]:
def normalize_state_value(x):
    import pandas as pd

    if pd.isna(x):
        return ""

    x = str(x).strip().lower()
    
    if len(x) == 2:
        return x.upper()

    return STATE_MAP.get(x, x.upper())

# Location interest matching

In [49]:
top_titles_df = title_scores.head(5).copy()

student_row = students_sample.iloc[student_id_to_check]
preferred_location_raw = student_row["Location Interest"]
preferred_state = normalize_state_value(preferred_location_raw)

print(f"Student: {student_row['Academic Major']} student")
print(f"Preferred location: {preferred_location_raw}\n")
print("Top Career Matches:\n")

for i, (_, row) in enumerate(top_titles_df.iterrows(), start=1):
    title = row["title"]
    fit_score = row["avg_similarity"]

    all_states = (
        jobs_text[jobs_text["normalized_title"] == title]
        .groupby("state", as_index=False)
        .size()
        .rename(columns={"size": "demand_count"})
        .sort_values("demand_count", ascending=False)
    )

    top_states = all_states.head(5)

    state_list = [
        f"{r['state']} ({r['demand_count']})"
        for _, r in top_states.iterrows()
    ]

    print(f"{i}. {title.title()}")
    print(f"   Fit Score: {fit_score:.4f}")
    print(f"   Top Demand Locations: {', '.join(state_list)}")
    print()

Student: marketing student
Preferred location: remote / flexible

Top Career Matches:

1. Marketing Manager
   Fit Score: 0.5148
   Top Demand Locations: CA (1)

2. Seo Associate | Bankrate
   Fit Score: 0.4585
   Top Demand Locations: NY (1)

3. Media Marketing Consultant Work
   Fit Score: 0.4543
   Top Demand Locations: MD (1)

4. Director Media
   Fit Score: 0.4066
   Top Demand Locations: NY (1)

5. Education Specialist
   Fit Score: 0.4044
   Top Demand Locations: TX (1)



Synthetic Data Generation - Tabular Format

We first generated synthetic job tables using two standard single-table tabular synthesizers, Gaussian Copula and CTGAN, trained on the real job postings dataset.

- how well each method preserves column distributions
- how well each preserves relationships between columns
- whether either is usable for downstream recommendation experiments

Can standard tabular synthetic data methods reproduce the structure of my job dataset?

Can I synthesize job descriptions good enough for BERT-based recommendation?

#SDV metadata helps the synthesizer understand the table structure
#pip install sdv sdmetrics pandas

from sdv.metadata import Metadata

 Two synthesizers:
 - CTGAN: deep learning / GAN-based
 - GaussianCopula: statistical baseline
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer

Evaluation tools from SDV
from sdv.evaluation.single_table import run_diagnostic, evaluate_quality

We will be using the jobs_sample data we saved as a csv file
To avoid modifying the original jobs_sample dataframe we created earlier, we will create a copy of it and work with that for synthetic data generation and testing our model later on.
data = jobs_sample.copy()

if "title" in data.columns and "description" in data.columns:
    data["job_text"] = data["title"].fillna("") + " " + data["description"].fillna("")
    data["job_text"] = data["job_text"].str.strip()

print("\nSample combined text:")
if "job_text" in data.columns:
    print(data["job_text"].head(3))

candidate_columns = [
    col for col in [
        "title",
        "description",
        "company_name",
        "location",
        "city",
        "state",
        "work_type",
        "views"
    ] if col in data.columns
]

model_data = data[candidate_columns].copy()

model_data.shape

#Metadata describes the table schema to SDV.
metadata = Metadata.detect_from_dataframe(data=model_data)
metadata.save_to_json("job_metadata.json", mode="overwrite")

print("\nMetadata:")
print(metadata.to_dict())

#TRAIN GAUSSIAN COPULA

gaussian = GaussianCopulaSynthesizer(metadata)
gaussian.fit(model_data)

synthetic_gaussian = gaussian.sample(num_rows=len(model_data))
synthetic_gaussian.to_csv("synthetic_gaussian.csv", index=False)


Tabular synthetic data tools can handle the structured part of your dataset, but they are not well suited to your raw text-heavy job data as-is.

Standard tabular synthetic data generators were not sufficient for the full job-posting dataset because high-cardinality text columns caused impractical preprocessing expansion and did not match the assumptions of tabular synthesis. This motivated a shift toward mixed-data generation, separating structured synthesis from text generation.

# Text aware  generation methods

We will be using LLM for this phase

In [50]:
# We will be using the jobs_sample data we saved as a csv file
#  We create a copy
data = jobs_sample.copy()

In [51]:
TARGET_COLUMNS = [
    "job_id",
    "company_name",
    "title",
    "description",
    "location",
    "company_id",
    "views",
    "skills_desc",
    "work_type",
    "zip_code",
    "fips",
    "city",
    "state"
]

JOB_SCHEMA = {
    "name": "synthetic_job_row",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "job_id": {"type": ["string", "null"]},
            "company_name": {"type": ["string", "null"]},
            "title": {"type": "string"},
            "description": {"type": ["string", "null"]},
            "location": {"type": ["string", "null"]},
            "company_id": {"type": ["string", "null"]},
            "views": {"type": ["number", "null"]},
            "skills_desc": {"type": ["string", "null"]},
            "work_type": {"type": ["string", "null"]},
            "zip_code": {"type": ["string", "null"]},
            "fips": {"type": ["string", "null"]},
            "city": {"type": ["string", "null"]},
            "state": {"type": ["string", "null"]}
        },
        "required": TARGET_COLUMNS,
        "additionalProperties": False
    }
}

In [52]:
from typing import Any, Optional

# HELPER FUNCTIONS

def safe_str(value: Any) -> Optional[str]:
    """Convert value to stripped string; return None if missing."""
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text != "" else None


def safe_float(value: Any) -> Optional[float]:
    """Convert value to float if possible; otherwise None."""
    if pd.isna(value):
        return None
    try:
        return float(value)
    except:
        return None


def clean_zip_code(value: Any) -> Optional[str]:
    """Keep ZIP as string, preserving leading zeros if possible."""
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def clean_fips(value: Any) -> Optional[str]:
    """Keep FIPS as string."""
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def generate_synthetic_job_id() -> str:
    """Create a synthetic job_id."""
    return uuid.uuid4().hex[:8]


def generate_synthetic_company_id() -> str:
    """Create a synthetic company_id."""
    return uuid.uuid4().hex[:10]


def truncate_text(text: Optional[str], max_len: int = 4000) -> Optional[str]:
    """Trim overly long generated text if needed."""
    if text is None:
        return None
    text = str(text).strip()
    return text[:max_len]


def preserve_missingness(original_value: Any, generated_value: Any) -> Any:
    """
    Optional rule:
    If the original cell was missing, keep it missing in the synthetic row.
    This helps preserve missingness patterns.
    """
    if pd.isna(original_value):
        return None
    return generated_value

In [53]:
def build_prompt(row: pd.Series) -> str:
    """
    We build a prompt that asks the LLM to generate one synthetic row
    with exactly the same schema.
    """

    example_input = {
        "job_id": safe_str(row.get("job_id")),
        "company_name": safe_str(row.get("company_name")),
        "title": safe_str(row.get("title")),
        "description": safe_str(row.get("description")),
        "location": safe_str(row.get("location")),
        "company_id": safe_str(row.get("company_id")),
        "views": safe_float(row.get("views")),
        "skills_desc": safe_str(row.get("skills_desc")),
        "work_type": safe_str(row.get("work_type")),
        "zip_code": safe_str(row.get("zip_code")),
        "fips": safe_str(row.get("fips")),
        "city": safe_str(row.get("city")),
        "state": safe_str(row.get("state")),
    }

    prompt = f"""
You are generating one synthetic job-posting record for research purposes.

Your task:
- Create a NEW synthetic row inspired by the input row.
- Keep the SAME JSON KEYS exactly.
- Do NOT copy the original text.
- Keep the row realistic and internally consistent.
- Preserve the same general style, data types, and meaning of each field.
- The output must be valid JSON only.
- If an input field is null, you may return null for that field.
- Keep title realistic.
- Keep description realistic and professional.
- Make location, city, state, zip_code, and fips logically consistent.
- Keep work_type realistic.
- Keep views numeric or null.
- Keep skills_desc as short skill-oriented text or null.

Input row:
{json.dumps(example_input, ensure_ascii=False, indent=2)}

Return JSON with exactly these keys:
{json.dumps(TARGET_COLUMNS, ensure_ascii=False)}
"""
    return prompt.strip()

In [57]:
# 6. LLM CALL FUNCTION

load_dotenv()

api_key = os.getenv("OPEN_AI_API_KEY")
if not api_key:
    raise ValueError("OPEN_AI_API_KEY not found in .env")

client = OpenAI(api_key=api_key)

In [60]:
def call_llm(prompt: str, model: str = "gpt-4o-mini", max_retries: int = 3) -> dict:
    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=model,
                input=[
                    {
                        "role": "system",
                        "content": (
                            "You generate synthetic job-posting rows for research. "
                            "Follow the provided schema exactly."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": JOB_SCHEMA["name"],
                        "strict": JOB_SCHEMA["strict"],
                        "schema": JOB_SCHEMA["schema"]
                    }
                },
                temperature=0.7,
                max_output_tokens=1200
            )

            return json.loads(response.output_text)

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(1.5)

    raise ValueError("Failed to generate valid JSON after retries.")

In [61]:
data = jobs_sample.copy()
row = data.iloc[0]

prompt = build_prompt(row)
synthetic_row = call_llm(prompt, model="gpt-4o-mini")

print(synthetic_row)

{'job_id': '4904567312', 'company_name': 'outdoor world', 'title': 'hunting sales associate - part time', 'description': 'position overview:\n\nthe hunting sales associate is responsible for providing exceptional customer service and support in the hunting department. this role involves assisting customers with their purchases, answering inquiries about hunting equipment, and ensuring a well-organized sales floor. duties include greeting customers promptly, offering guidance on products, and maintaining stock levels on the sales floor.\n\nkey responsibilities:\n\ncommits to delivering outstanding customer service to enhance the shopping experience. assists customers in finding suitable hunting gear by understanding their needs, recommending appropriate products, and promoting store initiatives. ensures compliance with all regulations regarding the sale of hunting equipment. maintains a clean and organized sales area and restocks merchandise as needed. stays updated on current promotion

In [62]:
synthetic_rows = []

for i, (_, row) in enumerate(jobs_sample.iterrows(), start=1):
    print(f"Generating row {i}...")

    try:
        prompt = build_prompt(row)
        synthetic_row = call_llm(prompt)
        synthetic_rows.append(synthetic_row)

    except Exception as e:
        print(f"Error on row {i}: {e}")

Generating row 1...
Generating row 2...
Generating row 3...
Generating row 4...
Generating row 5...
Generating row 6...
Generating row 7...
Generating row 8...
Generating row 9...
Generating row 10...
Generating row 11...
Generating row 12...
Generating row 13...
Generating row 14...
Generating row 15...
Generating row 16...
Generating row 17...
Generating row 18...
Generating row 19...
Generating row 20...
Generating row 21...
Generating row 22...
Generating row 23...
Generating row 24...
Generating row 25...
Generating row 26...
Generating row 27...
Generating row 28...
Generating row 29...
Generating row 30...
Generating row 31...
Generating row 32...
Generating row 33...
Generating row 34...
Generating row 35...
Generating row 36...
Generating row 37...
Generating row 38...
Generating row 39...
Generating row 40...
Generating row 41...
Generating row 42...
Generating row 43...
Generating row 44...
Generating row 45...
Generating row 46...
Generating row 47...
Generating row 48...
G

In [63]:
synthetic_df = pd.DataFrame(synthetic_rows, columns=TARGET_COLUMNS)

print("Preview:")
print(synthetic_df.head())

print("Shape:", synthetic_df.shape)

Preview:
       job_id                   company_name  \
0  3903466306                 bass pro shops   
1  5728391462          global legal services   
2  2891234567                     home depot   
3  3906097454                       bmo u.s.   
4  3903473958  university of central florida   

                                      title  \
0  outdoor equipment specialist - part time   
1                       executive assistant   
2                 inventory control manager   
3     senior manager, strategic initiatives   
4            building systems technician ii   

                                         description  \
0  position summary:\n\nthe outdoor equipment spe...   
1  a prestigious law firm located in seattle, wa,...   
2  we are seeking a dedicated inventory control m...   
3  reporting to the director of strategic initiat...   
4  department number/name: 0-0450-002 / facilitie...   

                    location company_id  views  \
0                lubbock, tx    

In [ ]:
# synthetic_df.to_csv("data/synthetic_jobs_openai.csv", index=False)

# # When it saves successfully, it will print the message below.
# print("Saved synthetic_jobs_openai.csv")    

In [64]:
synthetic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   job_id        1000 non-null   object 
 1   company_name  990 non-null    object 
 2   title         1000 non-null   object 
 3   description   1000 non-null   object 
 4   location      1000 non-null   object 
 5   company_id    990 non-null    object 
 6   views         1000 non-null   float64
 7   skills_desc   640 non-null    object 
 8   work_type     1000 non-null   object 
 9   zip_code      992 non-null    object 
 10  fips          922 non-null    object 
 11  city          990 non-null    object 
 12  state         998 non-null    object 
dtypes: float64(1), object(12)
memory usage: 101.7+ KB


Now we will use the synthetic_df to evaluate the performance of our recommendation system and compare it with the real data. 
We will follow the same steps as we did with the real data, we will build job text, vectorize it using BERT, and then compute 
similarity with student profiles to get recommendations. We will then compare the recommendations we get from the synthetic data 
with those we get from the real data to see how well the synthetic data is performing.